# 04 — The estimator RandomForestClassifier and its parameters

*Chapter 06 — Random Forests · Notebook 4 of 5*

You have built a random forest by hand, one idea at a time: bootstrap many trees and vote (NB 1),
let each split see only a random subset of features so the trees decorrelate (NB 2), and read the
forest's error for free from the points each tree left out (NB 3). Now we meet the real estimator,
`sklearn.ensemble.RandomForestClassifier`, and learn its dials — what each one does, how it fails,
and how to tune it honestly. The reassuring news first: there is nothing in the library you have not
already built.

**Prerequisites.**
- **This chapter, NB 1–3:** bagging (bootstrap + majority vote, and that a hand-bag equals a forest
  with feature subsampling switched off); the "random" — per-split feature subsampling and the
  pairwise tree correlation ρ it lowers; out-of-bag (OOB) estimation, the forest's free in-training
  score.
- **Chapter 04 (Decision Trees):** a single tree is **high-variance**, and impurity (MDI) feature
  importance — which is **biased** toward continuous / high-cardinality features (Strobl 2007).
- **Module 00:** the train/test split and leakage (NB 04); cross-validation for model selection
  (NB 10).

**What you'll be able to do.**
- Confirm that `RandomForestClassifier(max_features=None)` is exactly the bagging you built.
- Set `n_estimators` by reading the out-of-bag error curve (and know more trees never overfit).
- Use `max_features` — the decorrelation dial — and know why `'sqrt'` is a robust default.
- Explain why a forest grows deep trees on purpose and tolerates it.
- Read forest feature importances, name their bias, and know permutation importance as the cross-check.
- Tune with `GridSearchCV` on the training set and report a single sealed-test number.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

# breast cancer: 30 (correlated) features, malignant = 1 (the costly miss).
data = load_breast_cancer(as_frame=True)
X = data.data
y = (data.target == 0).astype(int)  # sklearn target 0 = malignant -> encode malignant as 1
feature_names = list(X.columns)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"train: {X_train.shape[0]} patients x {X_train.shape[1]} features"
      f"     test: {X_test.shape[0]} patients")
print(f"malignant -- train {y_train.sum()}/{len(y_train)}, test {y_test.sum()}/{len(y_test)}")

# The estimator's defaults, straight from the library.
d = RandomForestClassifier()
print(f"\ndefaults: n_estimators={d.n_estimators}, max_features={d.max_features!r}, "
      f"bootstrap={d.bootstrap}, oob_score={d.oob_score}, max_depth={d.max_depth}, "
      f"criterion={d.criterion!r}")

## The estimator is the two ideas you already built

A random forest is two moves stacked on a decision tree: train each tree on its own **bootstrap**
resample and average the votes (bagging, NB 1), and let each split choose among only a **random
subset** of the features (the "random", NB 2). `RandomForestClassifier` is precisely that, wrapped in
one class with a handful of dials.

The clearest way to see this is to switch the second idea **off**. With `max_features=None`, every
split may consider all features — there is no subsampling left, so the forest is plain bagging. It
should reproduce the hand-bag from NB 1. Let's check, then turn the dials back on.

In [ ]:
# Hand-bagging (NB 1): B deep trees, each on its own bootstrap, combined by majority vote.
# We fix one base-learner seed (the bootstraps still vary tree to tree); with max_features off,
# this reproduces RandomForestClassifier(max_features=None).
def hand_bag_predict(X_tr, y_tr, X_eval, n_trees=200, seed=SEED):
    n = X_tr.shape[0]
    boot_rng = np.random.default_rng(seed)
    votes = np.zeros((X_eval.shape[0], n_trees))
    for b in range(n_trees):
        idx = boot_rng.integers(0, n, n)  # a bootstrap resample
        tree_b = DecisionTreeClassifier(random_state=seed).fit(X_tr[idx], y_tr[idx])
        votes[:, b] = tree_b.predict(X_eval)
    return (votes.mean(axis=1) >= 0.5).astype(int)  # majority vote


Xtr_a, ytr_a, Xte_a = X_train.to_numpy(), y_train.to_numpy(), X_test.to_numpy()
hand_pred = hand_bag_predict(Xtr_a, ytr_a, Xte_a, n_trees=200)

rf_bag = RandomForestClassifier(n_estimators=200, max_features=None, random_state=SEED)
rf_bag.fit(X_train, y_train)
rf_default = RandomForestClassifier(n_estimators=200, random_state=SEED).fit(X_train, y_train)

print(f"hand-bag (200 trees, all features):        test {accuracy_score(y_test, hand_pred):.4f}")
print(f"RandomForestClassifier(max_features=None): test "
      f"{accuracy_score(y_test, rf_bag.predict(X_test)):.4f}")
print(f"RandomForestClassifier() default ('sqrt'): test "
      f"{accuracy_score(y_test, rf_default.predict(X_test)):.4f}")

**Read the parity.** Switch feature subsampling off and the library lands on the **same** test
accuracy as the bag we built by hand (0.9357): `max_features=None` *is* bagging — bootstrap, grow
deep trees, vote. (This is an accuracy match at 200 trees, not a tree-for-tree clone: the two draw
different bootstraps, and as NB 1 noted, a one-tree forest is not a lone decision tree because the
splitter randomises tie-breaks.) Turn subsampling back on — the default `'sqrt'`, about 5 of 30
features per split — and accuracy edges up to **0.9415**: with 30 features to choose among, the
"random" decorrelates the trees and the forest improves. On NB 1's two-feature moons it went the
other way (`'sqrt'` there meant 1 feature per split, which *starved* the trees: 0.900 < 0.933) —
feature subsampling pays only when there are many features to subsample.

## `n_estimators`: how many trees?

Each tree casts a vote; more votes steady the average, exactly the variance reduction of NB 1 (the
variance of an average shrinks like σ²/B). So how many trees are enough? We need no separate
validation set to find out — the **out-of-bag** error (NB 3) reads the forest's generalization for
free as the trees accumulate. Let's watch it fall, alongside the sealed-test error.

In [ ]:
# OOB error and test error as the forest grows. scikit-learn warns at very few trees
# (NB 3: some points are then never out-of-bag) -- we let the warning through, not hide it.
tree_counts = [1, 5, 10, 25, 50, 100, 300, 500]
oob_err, test_err = [], []
for ne in tree_counts:
    rf = RandomForestClassifier(n_estimators=ne, oob_score=True, random_state=SEED)
    rf.fit(X_train, y_train)
    oob_err.append(1 - rf.oob_score_)
    test_err.append(1 - accuracy_score(y_test, rf.predict(X_test)))

fig = viz.plot_train_test_curve(
    tree_counts, oob_err, test_err,
    xlabel="number of trees (n_estimators)", ylabel="error",
)
ax = fig.axes[0]
ax.set_xscale("log")
ax.legend(["out-of-bag error", "test error"])
print("OOB error :", [round(float(e), 3) for e in oob_err])
print("test error:", [round(float(e), 3) for e in test_err])

**Read the figure.** Both curves drop steeply over the first ~25 trees, then **flatten**: past
a few dozen trees the out-of-bag error settles near 0.04 and the test error hovers around 0.05–0.06,
and neither **climbs** as we keep adding trees. This is the key fact about `n_estimators`: more trees
**never overfit** — they only steady the estimate, at a cost in compute and memory. So pick "enough"
(a hundred or a few hundred), not "as many as possible". At the far-left, few-tree end the out-of-bag
estimate is wild and scikit-learn **warns**: with so few trees, some training points were left in-bag
by every one of them and have no out-of-bag grader at all (NB 3). We let that warning print — it is
the library being honest, not a failure.

## `max_features`: the decorrelation dial

This is the dial that turns a bag of trees into a *random* forest. At each split, a tree may consider
only `max_features` of the inputs, chosen at random. Allow **all** of them and every tree chases the
same dominant splits — they correlate, and averaging helps less (NB 2's ρ was high). Allow only a
**few** and the trees are forced down different paths — they decorrelate (ρ falls), and the average is
steadier. The classification default is **`'sqrt'`**: about √30 ≈ 5 features per split here.

In [ ]:
# Sweep max_features. We read OOB and cross-validated accuracy (NOT test accuracy: on 171 test
# points the per-setting ranking is seed-fragile, NB 2). The story is the picture, not a winner.
settings = [1, 2, "sqrt", "log2", 10, 20, 30]
print(f"{'max_features':>13}  {'OOB acc':>8}  {'CV mean':>8}  {'CV std':>7}")
for mf in settings:
    rf = RandomForestClassifier(
        n_estimators=200, max_features=mf, oob_score=True, random_state=SEED
    ).fit(X_train, y_train)
    cv_scores = cross_val_score(
        RandomForestClassifier(n_estimators=200, max_features=mf, random_state=SEED),
        X_train, y_train, cv=cv,
    )
    m, sd = cv_scores.mean(), cv_scores.std()
    print(f"{str(mf):>13}  {rf.oob_score_:>8.3f}  {m:>8.3f}  {sd:>7.3f}")

**Read the result.** Two honest readings. First, across the whole range — from 1 feature per
split to all 30 — accuracy barely moves (out-of-bag and cross-validated accuracy both sit around
0.95–0.96). A random forest is **forgiving**: the default `'sqrt'` is a safe choice you will rarely
need to touch. Second, the dial's real *job* is decorrelation, which we measured directly in NB 2: ρ
rose from about 0.70 at one feature per split to 0.82 with all features. All-features
(`max_features=30`) is plain bagging — the highest ρ, the least decorrelation, the setting the
"random" was invented to beat. Here the accuracy payoff of subsampling is small because the problem is
easy and the features are informative; the dial **bites hardest** when features are many and
correlated — which is exactly the demanding case waiting in NB 5.

## `max_depth` and `min_samples_leaf`: why a forest grows deep

In chapter 04, a lone tree grown without limit **overfit**: it drove its training error to zero while
its test accuracy sagged (the U-curve). A random forest does the opposite of what that lesson might
suggest — it grows each tree to **full depth on purpose**. The reason is NB 1: each deep tree is a
low-bias, high-variance learner, and averaging cancels the variance. So `RandomForestClassifier`
defaults to `max_depth=None` and `min_samples_leaf=1`. Let's watch a single tree and a forest as we
let depth grow.

In [ ]:
# A single tree vs a 200-tree forest as we allow deeper trees.
depths = [1, 2, 3, 4, 5, 6, 8, 12]
tree_test, forest_test = [], []
for md_ in depths:
    tree = DecisionTreeClassifier(max_depth=md_, random_state=SEED).fit(X_train, y_train)
    forest = RandomForestClassifier(
        n_estimators=200, max_depth=md_, random_state=SEED
    ).fit(X_train, y_train)
    tree_test.append(accuracy_score(y_test, tree.predict(X_test)))
    forest_test.append(accuracy_score(y_test, forest.predict(X_test)))

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(depths, tree_test, marker="s", color=COLORS["highlight"], linewidth=2, label="single tree")
ax.plot(depths, forest_test, marker="o", color=COLORS["model"], linewidth=2,
        label="forest (200 trees)")
ax.set_xlabel("max_depth")
ax.set_ylabel("test accuracy")
ax.legend()
fig.tight_layout()

# At full depth both memorize the training set -- the tree's high test variance is the difference.
print(f"at max_depth=12: single tree train "
      f"{accuracy_score(y_train, tree.predict(X_train)):.3f} / test {tree_test[-1]:.3f}"
      f"     forest train {accuracy_score(y_train, forest.predict(X_train)):.3f}"
      f" / test {forest_test[-1]:.3f}")

# Run-to-run stability: a single deep tree (on bootstraps) vs the forest, across 20 seeds.
tree_accs, forest_accs = [], []
boot_rng = np.random.default_rng(SEED)
nrow = X_train.shape[0]
for s in range(20):
    idx = boot_rng.integers(0, nrow, nrow)
    t = DecisionTreeClassifier(random_state=s).fit(Xtr_a[idx], ytr_a[idx])
    tree_accs.append(accuracy_score(y_test, t.predict(Xte_a)))
    f = RandomForestClassifier(n_estimators=200, random_state=s).fit(X_train, y_train)
    forest_accs.append(accuracy_score(y_test, f.predict(X_test)))
print(f"run-to-run std (20 seeds): single deep tree {np.std(tree_accs):.4f}"
      f"   forest {np.std(forest_accs):.4f}")
print(f"mean accuracy: single tree {np.mean(tree_accs):.3f}   forest {np.mean(forest_accs):.3f}")

**Read the figure.** Follow the single-tree line (in the highlight colour): as we allow more
depth its test accuracy climbs, then **wobbles and sags** — by full depth it has memorized the
training set (training accuracy 1.0) and generalizes unevenly, around 0.90. The forest line (in the
model colour) grows the *same* full-depth trees, yet its test accuracy rises and then **stays flat and
high**, around 0.94. Averaging cancels the variance that sinks a lone deep tree — so the forest keeps
the trees' full flexibility without paying for it. The stability numbers say the same thing: across 20
reseeds the single deep tree's accuracy swings with a standard deviation of about **0.016**, the
forest's only about **0.004** — roughly four times steadier, and more accurate on average. This is why
a forest leaves `max_depth=None` and `min_samples_leaf=1`: deep trees are a feature, not a bug, once
you average them.

## The remaining knobs, in one breath

Three more parameters round out the estimator, and their defaults are already sensible:

- **`bootstrap`** (default `True`) is the bagging itself. Set it to `False` and every tree trains on
  the full dataset; the only randomness left is the per-split feature subsampling, so the trees barely
  differ and you lose most of the variance reduction. Leave it on.
- **`class_weight`** re-weights the classes so a rare class is not drowned out by a common one — the
  tool for imbalance, which the demanding case in NB 5 will need.
- **`n_jobs`** sets how many cores train trees in parallel. The trees are independent, so a forest is
  *embarrassingly parallel*: `n_jobs=-1` uses every core and changes the timing, never the result.

## Feature importance, stabilized

Chapter 04 read a single tree's feature importance from how much each split reduced impurity (the MDI,
or mean decrease in impurity). The trouble was instability: a single tree poured almost all of its
importance onto whichever feature it happened to split on first. A forest averages MDI over hundreds
of trees, each grown on a different bootstrap and a different feature subset — so the read **steadies**
and **spreads** across the features that genuinely carry signal. Let's compare a single tree's
importance to the forest's.

In [ ]:
single_tree = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(X_train, y_train)

fig, (ax_tree, ax_forest) = plt.subplots(1, 2, figsize=(13.5, 5))
viz.plot_feature_importances(
    feature_names, single_tree.feature_importances_, top=8, ax=ax_tree, color=COLORS["highlight"]
)
ax_tree.set_title("single tree: one spike")
viz.plot_feature_importances(
    feature_names, forest.feature_importances_, top=8, ax=ax_forest, color=COLORS["model"]
)
ax_forest.set_title("forest of 300: the credit spreads")
fig.tight_layout()

top_tree = feature_names[int(np.argmax(single_tree.feature_importances_))]
print(f"single tree: '{top_tree}' alone scores {single_tree.feature_importances_.max():.3f}")
print(f"forest: top importance is only {forest.feature_importances_.max():.3f}"
      f"   ({int((forest.feature_importances_ >= 0.05).sum())} features score >= 0.05)")

**Read the figure.** The single tree (left) put about **0.74** of all its importance on one
feature, `mean concave points` — an artefact of which of several correlated measurements it happened
to split on first. The forest (right) tells a fairer story: its top importance is only about **0.15**
— note the two panels carry **different** x-axes, so that 0.15 is about a *fifth* of the tree's 0.74
spike, not a bar of similar length. The credit is now **spread** across the whole correlated group of
`concave points`, `perimeter`, `radius` and `area` measurements that each carry the
malignant-vs-benign signal. Averaging did not change the biology; it removed the single tree's
arbitrary spike.

Two honest cautions carry forward. MDI is still **biased** — it favours continuous and high-cardinality
features, and it **dilutes** a signal across a group of correlated features, so no single bar can be
read as "the" important one; read at the group level (Strobl 2007). The honest cross-check is
**permutation importance**: shuffle one column, refit nothing, and measure how far the accuracy drops
— a direct test of how much the model *relies* on that feature. We put permutation importance to work
on the demanding case in NB 5.

## Honest tuning

Module 00's discipline holds here as everywhere: choose hyperparameters by **cross-validation on the
training set**, then read the **sealed test once**. (The out-of-bag score is a cheap stand-in for a
quick check, but for a number you will report, use cross-validation and a held-out test.) We search a
small grid over the three dials that matter — `max_features`, `min_samples_leaf`, `max_depth` — and
compare the tuned forest to the default.

In [ ]:
grid = {
    "max_features": ["sqrt", "log2", 0.5],
    "min_samples_leaf": [1, 2, 5],
    "max_depth": [None, 10],
}
search = GridSearchCV(
    RandomForestClassifier(n_estimators=300, random_state=SEED), grid, cv=cv, n_jobs=-1
)
search.fit(X_train, y_train)

default_cv = cross_val_score(
    RandomForestClassifier(n_estimators=300, random_state=SEED), X_train, y_train, cv=cv
).mean()
default_rf = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(X_train, y_train)

print(f"tuned forest:   best params {search.best_params_}")
print(f"                CV {search.best_score_:.3f}  ->  sealed test "
      f"{accuracy_score(y_test, search.predict(X_test)):.3f}")
print(f"default forest: CV {default_cv:.3f}  ->  sealed test "
      f"{accuracy_score(y_test, default_rf.predict(X_test)):.3f}")

**Read the result.** Tuning nudged the cross-validated accuracy from about **0.955** to
**0.957** and the sealed test from **0.942** to **0.947** — a fraction of a point. The default forest
was already strong, and that is the headline about random forests as much as any single number: they
are **strong with almost no tuning** (and, inheriting from trees, with no feature scaling — you
confirm that in *Your turn*). Notice what the search chose: `max_depth=None` and `min_samples_leaf=1`
— it kept the trees deep, agreeing with the figure above that a forest wants full-depth trees.

## Your turn

1. **Read the OOB curve (easy).** Re-fit the forest for `n_estimators` in `[5, 25, 100, 300, 1000]`
   with `oob_score=True`, and print the out-of-bag accuracy each time. Name the smallest forest whose
   out-of-bag score has stopped improving in any meaningful way — that is your "enough".

2. **Feel the decorrelation dial (medium).** Fit two forests, one with `max_features=None` (the
   bagging end) and one with `max_features='sqrt'`, and compare their out-of-bag accuracy. Using NB 2's
   ρ, explain which one grows the *more correlated* trees, and why that is the one expected to gain
   less from averaging.

3. **Scale-invariance (harder).** Standardize the features (subtract the mean, divide by the standard
   deviation) and fit a forest on the standardized data with the same seed. Compare its test
   predictions to a forest fit on the raw data — they should be **identical**. Explain why, using what
   chapter 04 taught about how a tree chooses a split. (This is why no `StandardScaler` appears in this
   chapter.)

## What you built

- You confirmed that `RandomForestClassifier(max_features=None)` reaches the **same** accuracy as the
  hand-bag from NB 1 — the library is the two ideas you already built.
- You read the **`n_estimators`** curve from the out-of-bag error and saw that more trees bring
  **diminishing returns** and **never overfit** — pick "enough", not "as many as possible".
- You turned **`max_features`**, the decorrelation dial, and learned that a forest is **forgiving**
  (`'sqrt'` a robust default) while the dial's real effect — lowering ρ — was measured in NB 2.
- You saw why a forest grows **deep** trees on purpose and tolerates it: averaging cancels the variance
  that sinks a lone deep tree (about four times steadier here).
- You read forest **feature importance** — the credit **spreads** across correlated features where a
  single tree spiked — and named its **bias**, with **permutation importance** as the honest
  cross-check (NB 5).
- You tuned with **`GridSearchCV`** on the training set and read one sealed test — and found the
  default forest was already strong.

**Vocabulary you now own:** `n_estimators` · `max_features` (the decorrelation dial) ·
`max_depth` / `min_samples_leaf` · `bootstrap` / `class_weight` / `n_jobs` · MDI vs permutation
importance · out-of-bag-guided model selection.

## Going further (optional)

- **`ExtraTrees`** (extremely randomized trees) push the "random" one step further: they also pick
  split *thresholds* at random, trading a little more bias for even lower variance and faster fits.
- **`warm_start=True`** lets you add trees to an existing forest without refitting from scratch —
  handy when reading an `n_estimators` curve.
- **Importance, honestly.** We *named* permutation importance here; NB 5 uses it on a harder problem,
  where impurity importance and permutation importance can disagree and the difference matters.

## References

- Breiman, L. (2001). *Random Forests.* Machine Learning 45, 5–32.
  DOI: [10.1023/A:1010933404324](https://doi.org/10.1023/A:1010933404324)
- Breiman, L. (1996). *Bagging predictors.* Machine Learning 24, 123–140.
  DOI: [10.1007/BF00058655](https://doi.org/10.1007/BF00058655)
- Ho, T. K. (1998). *The random subspace method for constructing decision forests.* IEEE TPAMI 20(8),
  832–844. DOI: [10.1109/34.709601](https://doi.org/10.1109/34.709601)
- Strobl, C., Boulesteix, A.-L., Zeileis, A., & Hothorn, T. (2007). *Bias in random forest variable
  importance measures.* BMC Bioinformatics 8, 25.
  DOI: [10.1186/1471-2105-8-25](https://doi.org/10.1186/1471-2105-8-25)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §15 (Random Forests). DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning*
  (2nd ed.), §8.2. DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1)

*Previous: 03 — Out-of-bag estimation. Next: 05 — A demanding case: covtype.*